In [25]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
import os
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
from groq import Groq

In [26]:
MODEL = "llama-3.1-8b-instant"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

In [27]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

In [28]:
retriever = vectorstore.as_retriever()

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

In [29]:
retriever.invoke("Who is Avery?")

[Document(id='3b323192-5bec-4d8d-964b-07cff7a79c44', metadata={'source': 'C:\\Users\\Rana\\projectt\\llm\\RAG\\knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market trends and consumer preferences in the insurance space. This position laid the groundwork for Avery’s future entrepreneurial endeavors.\n\n## Annual Performance History\n- **2015**: **Exceeds Expectations**  \n  Avery’s leadership during Insurellm's foundational year led to successful product launches and securing initial funding."),
 Document(id='65b0f08d-ca0f-4985-a8e8-b446a4ce8e8a', metadata={'source': 'C:\\Users\\Rana\\projectt\\llm\\RAG\\knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content='- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leadin

In [30]:
llm.invoke("Who is Avery?")

AIMessage(content='There are several notable individuals named Avery. Here are a few:\n\n1. **Avery Bradley**: An American professional basketball player who plays in the NBA. He has played for several teams, including the Boston Celtics, Detroit Pistons, and Los Angeles Clippers.\n2. **Avery Bradley (musician)**: An American musician and producer who has worked with artists such as Kanye West and Drake.\n3. **Avery Bradley (footballer)**: An American soccer player who has played for several teams, including the Seattle Sounders FC.\n4. **Avery Bradley (author)**: An American author who has written several books, including young adult fiction and non-fiction.\n5. **Avery Bradley (TV personality)**: An American TV personality who has appeared on several reality TV shows, including "The Bachelor" and "The Bachelorette".\n6. **Avery Bradley (music)**: Avery Bradley is also a music artist, who has released several albums and singles.\n\nHowever, without more context, it\'s difficult to det

In [32]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [33]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [34]:
answer_question("Who is Averi Lancaster?", [])

"I couldn't find any information about Averi Lancaster in the given context. However, I do have information about Avery Lancaster, who is the Co-Founder & CEO of Insurellm."

In [35]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
